# 2. Prepare analysis dataset

Versione pulita: setup unico del progetto, output tabellari invariati in `data/processed/analysis`.

In [1]:
from pathlib import Path
import sys
import pandas as pd

def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() or (candidate / ".git").exists() or (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Project root not found. Run the notebook from inside the project repository.")

def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path

def report_dir(root: Path, analysis_name: str, subfolder: str | None = None) -> Path:
    base = root / "reports" / analysis_name
    return ensure_dir(base if subfolder is None else base / subfolder)

ROOT = find_project_root()
print("ROOT:", ROOT)


ROOT: /home/eleonorab/repository/convective_cooling


In [ ]:
# =====================
# CONFIG
# =====================

INPUT_DIR = ROOT / 'data' / 'processed' / 'aggregated_by_signal'
OUTPUT_DIR = ROOT / 'data' / 'processed' / 'analysis'

ensure_dir(OUTPUT_DIR)

# =====================
# PHASE DEFINITION (min)
# =====================

def assign_phase(elapsed_min):
    if elapsed_min < 10:
        return "standing_1"
    elif elapsed_min < 20:
        return "walk_low_1"
    elif elapsed_min < 30:
        return "walk_mod_1"
    elif elapsed_min < 40:
        return "walk_low_2"
    elif elapsed_min < 50:
        return "walk_mod_2"
    else:
        return "standing_2"

# =====================
# LOAD ALL SIGNAL FILES
# =====================

all_dfs = []

for file in INPUT_DIR.glob("*.csv"):
    df = pd.read_csv(file)

    # Extract signal name from filename
    signal_name = file.stem.lower()  # es: t_abdomen

    df["signal"] = signal_name

    all_dfs.append(df)

# Merge all
df_long = pd.concat(all_dfs, ignore_index=True)

# =====================
# CLEAN + FEATURES
# =====================

# Ensure correct types
df_long["elapsed_minutes"] = df_long["elapsed_minutes"].astype(float)

# Assign phase
df_long["phase"] = df_long["elapsed_minutes"].apply(assign_phase)

# 5-min bins
df_long["time_bin_5min"] = (df_long["elapsed_minutes"] // 5) * 5

# =====================
# SAVE LONG FORMAT
# =====================

df_long.to_csv(OUTPUT_DIR / "all_signals_long.csv", index=False)

# =====================
# CREATE WIDE DATASET
# =====================

df_wide = df_long.pivot_table(
    index=["subject_id", "condition", "timestamp", "elapsed_seconds", "elapsed_minutes", "phase", "time_bin_5min"],
    columns="signal",
    values="value"
).reset_index()

# =====================
# ADD DERIVED VARIABLES
# =====================

# Microclimate gradients
if "t_microclimateback" in df_wide.columns and "t_microclimatefront" in df_wide.columns:
    df_wide["t_microclimate_gradient"] = (
        df_wide["t_microclimateback"] - df_wide["t_microclimatefront"]
    )

if "rh_microclimateback" in df_wide.columns and "rh_microclimatefront" in df_wide.columns:
    df_wide["rh_microclimate_gradient"] = (
        df_wide["rh_microclimateback"] - df_wide["rh_microclimatefront"]
    )

# =====================
# SAVE WIDE DATASET
# =====================

df_wide.to_csv(OUTPUT_DIR / "analysis_dataset_wide.csv", index=False)

print("Analysis dataset created successfully")

/tmp/ipykernel_252780/3069691530.py:45: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_long = pd.concat(all_dfs, ignore_index=True)


Analysis dataset created successfully
